#### 1. Initialization and Imports

Import only the core libraries needed for initialization, reproducibility, dataset loading, and readable tabular displays. Libraries used much later are imported in the section where they first become necessary.


In [ ]:
import random

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


#### 2. Hyperparameters

Keep the experiment settings together so a new run can be reproduced. `EPOCHS` is the configured maximum; training may be stopped manually after a completed epoch if validation performance has converged or begun to worsen.


In [ ]:
# Reproducibility (used in Sections 2, 8, 13, 19, and 21)
SEED = 42

# Dataset split and tokenization (used in Sections 8, 11, 13, 19, 21, and 23)
VALIDATION_SIZE = 0.10
MAX_LENGTH = 256
BATCH_SIZE = 16

# Training (used in Sections 15, 16, and 21)
EPOCHS = 50
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01

# Loss (used in Sections 15 and 21)
USE_WEIGHTED_LOSS = False
O_LABEL_WEIGHT = 1.0
PII_LABEL_WEIGHT = 2.0

# Learning-rate scheduler (used in Sections 15, 16, and 21)
USE_LR_SCHEDULER = True
LR_SCHEDULER_FACTOR = 0.5
LR_SCHEDULER_PATIENCE = 2
MIN_LEARNING_RATE = 1e-6

# ONNX comparison (used in Section 23)
ONNX_WARMUP_BATCHES = 3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Reproducibility seed: {SEED}")


#### 3. Dataset and Model Configuration

Store the required Hugging Face dataset and pretrained TinyBERT identifiers in one place. The model is identified here but is not loaded until Section 14.


In [ ]:
DATASET_NAME = "ai4privacy/pii-masking-openpii-1.5m"
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"


#### 4. Load Dataset

Load the configured Hugging Face dataset and its existing splits. This is intentionally separate because the dataset is large and may take time to download or restore from cache.


In [ ]:
dataset = load_dataset(DATASET_NAME)

dataset


#### 5. Dataset Citation and License

This notebook uses the **OpenPII 1.5M: Multilingual PII Masking Dataset with Asia Pacific Extension**, published by **Ai4Privacy / Ai Suisse SA** on Hugging Face.

The dataset contains synthetic personally identifiable information (PII) and is released under the **CC BY 4.0** license. It may be used for research, commercial use, redistribution, and modification, provided appropriate attribution is given.

**Dataset:** `ai4privacy/pii-masking-openpii-1.5m`  
**Publisher:** Ai4Privacy / Ai Suisse SA  
**License:** CC BY 4.0  
**Year:** 2026  
**Source:** https://huggingface.co/datasets/ai4privacy/pii-masking-openpii-1.5m

##### BibTeX

```bibtex
@dataset{ai4privacy_openpii_1_5m_2026,
  author = {Ai4Privacy},
  title = {OpenPII 1.5M: Multilingual PII Masking Dataset with Asia Pacific Extension},
  year = {2026},
  publisher = {Hugging Face},
  url = {https://huggingface.co/datasets/ai4privacy/pii-masking-openpii-1.5m}
}
```


#### 6. Filter English Data

Filter every original split using only `language == "en"`. No broad cleaning rule is applied, so English rows with an empty `privacy_mask` remain part of the experiment.


In [ ]:
english_dataset = dataset.filter(
    lambda example: example["language"] == "en"
)

print("Original dataset:")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data):,}")

print()
print("English-only dataset:")
for split_name, split_data in english_dataset.items():
    print(f"  {split_name}: {len(split_data):,}")


#### 7. Show Five Random Examples

Display source text beside the dataset's expected masked text. This inspection cell deliberately does not reset `SEED` or create a fixed sampling generator, so rerunning only this cell draws another sample.


In [ ]:
sample_size = min(5, len(english_dataset["train"]))

sample_indices = random.sample(
    range(len(english_dataset["train"])),
    k=sample_size,
)

random_examples = english_dataset["train"].select(sample_indices)

examples_df = pd.DataFrame({
    "Source text": random_examples["source_text"],
    "Masked text": random_examples["masked_text"],
})

pd.set_option("display.max_colwidth", None)

examples_df


#### 8. Create Train, Validation, and Final Test Splits

Split only the English portion of the original training split into project training and project validation data. Keep the English original validation split untouched as the final test set.


In [ ]:
train_validation_split = english_dataset["train"].train_test_split(
    test_size=VALIDATION_SIZE,
    seed=SEED,
    shuffle=True,
)

project_train_dataset = train_validation_split["train"]
project_validation_dataset = train_validation_split["test"]
final_test_dataset = english_dataset["validation"]

assert (
    len(project_train_dataset) + len(project_validation_dataset)
    == len(english_dataset["train"])
)

print(f"Project training rows:   {len(project_train_dataset):,}")
print(f"Project validation rows: {len(project_validation_dataset):,}")
print(f"Untouched final-test rows: {len(final_test_dataset):,}")


#### 9. Build the BIO Label List

Derive the entity ontology from the complete English original training split. Then inspect only the entity-type sets in project validation and final test. An unseen type raises a clear error instead of being removed, renamed, or mapped to a generic label.


In [ ]:
entity_types_by_split = {}

splits_for_entity_check = {
    "English original training": english_dataset["train"],
    "Project validation": project_validation_dataset,
    "Final test": final_test_dataset,
}

for split_name, split_data in splits_for_entity_check.items():
    split_entity_types = set()

    for row_index, privacy_mask in enumerate(split_data["privacy_mask"]):
        if not isinstance(privacy_mask, list):
            raise ValueError(
                f"{split_name} row {row_index} has a privacy_mask that is not a list."
            )

        for span_index, span in enumerate(privacy_mask):
            if not isinstance(span, dict):
                raise ValueError(
                    f"{split_name} row {row_index}, span {span_index} is not a dictionary."
                )

            entity_type = span.get("label")
            if not isinstance(entity_type, str) or not entity_type:
                raise ValueError(
                    f"{split_name} row {row_index}, span {span_index} has no valid label."
                )

            split_entity_types.add(entity_type)

    entity_types_by_split[split_name] = split_entity_types

training_entity_types = entity_types_by_split["English original training"]
unseen_validation_types = (
    entity_types_by_split["Project validation"] - training_entity_types
)
unseen_final_test_types = (
    entity_types_by_split["Final test"] - training_entity_types
)

if unseen_validation_types:
    raise ValueError(
        "Project validation contains unseen entity types: "
        f"{sorted(unseen_validation_types)}"
    )

if unseen_final_test_types:
    raise ValueError(
        "Final test contains unseen entity types: "
        f"{sorted(unseen_final_test_types)}"
    )

entity_types = sorted(training_entity_types)
label_list = ["O"]

for entity_type in entity_types:
    label_list.extend([f"B-{entity_type}", f"I-{entity_type}"])

label2id = {label: label_id for label_id, label in enumerate(label_list)}
id2label = {label_id: label for label, label_id in label2id.items()}
pii_label_ids = [label2id[label] for label in label_list if label != "O"]

print(f"Entity types from English original training: {len(entity_types)}")
print(f"BIO labels including O: {len(label_list)}")
print("Validation and final-test entity-type checks: passed")
print(label_list)


#### 10. Convert `privacy_mask` Spans into BIO Labels

The dataset stores PII as character spans. The focused helper below validates each annotation, preserves its entity type, and attaches the corresponding beginning and inside BIO labels. Empty masks are valid and return an empty list.


In [ ]:
def privacy_mask_to_bio_spans(source_text, privacy_mask, row_identifier):
    if not isinstance(source_text, str):
        raise ValueError(f"Row {row_identifier} has a source_text that is not a string.")

    if not isinstance(privacy_mask, list):
        raise ValueError(f"Row {row_identifier} has a privacy_mask that is not a list.")

    bio_spans = []

    for span_index, span in enumerate(privacy_mask):
        if not isinstance(span, dict):
            raise ValueError(
                f"Row {row_identifier}, span {span_index} is not a dictionary."
            )

        required_fields = {"value", "start", "end", "label"}
        missing_fields = required_fields - set(span)
        if missing_fields:
            raise ValueError(
                f"Row {row_identifier}, span {span_index} is missing fields: "
                f"{sorted(missing_fields)}"
            )

        span_value = span["value"]
        span_start = span["start"]
        span_end = span["end"]
        entity_type = span["label"]

        if type(span_start) is not int or type(span_end) is not int:
            raise ValueError(
                f"Row {row_identifier}, span {span_index} has non-integer bounds."
            )

        if not 0 <= span_start < span_end <= len(source_text):
            raise ValueError(
                f"Row {row_identifier}, span {span_index} has invalid bounds "
                f"({span_start}, {span_end}) for text length {len(source_text)}."
            )

        if not isinstance(span_value, str):
            raise ValueError(
                f"Row {row_identifier}, span {span_index} has a non-string value."
            )

        if source_text[span_start:span_end] != span_value:
            raise ValueError(
                f"Row {row_identifier}, span {span_index} does not match source_text."
            )

        if not isinstance(entity_type, str) or entity_type not in training_entity_types:
            raise ValueError(
                f"Row {row_identifier}, span {span_index} has unknown entity type "
                f"{entity_type!r}."
            )

        bio_spans.append({
            "value": span_value,
            "start": span_start,
            "end": span_end,
            "entity_type": entity_type,
            "begin_label": f"B-{entity_type}",
            "inside_label": f"I-{entity_type}",
        })

    bio_spans.sort(key=lambda span: (span["start"], span["end"]))

    for previous_span, current_span in zip(bio_spans, bio_spans[1:]):
        if current_span["start"] < previous_span["end"]:
            raise ValueError(
                f"Row {row_identifier} contains overlapping privacy_mask spans."
            )

    return bio_spans


In [ ]:
annotated_example_index = next(
    index
    for index, privacy_mask in enumerate(project_train_dataset["privacy_mask"])
    if privacy_mask
)

annotated_example = project_train_dataset[annotated_example_index]
example_bio_spans = privacy_mask_to_bio_spans(
    annotated_example["source_text"],
    annotated_example["privacy_mask"],
    annotated_example["uid"],
)

pd.DataFrame(
    [
        {
            "text span": span["value"],
            "character range": f"{span['start']}:{span['end']}",
            "beginning label": span["begin_label"],
            "inside label": span["inside_label"],
        }
        for span in example_bio_spans
    ]
)


#### 11. Tokenize and Align Labels

Use a fast TinyBERT tokenizer so character offsets are available. For each token, the first token or subword overlapping an entity receives `B-ENTITY`; later tokens or subwords for the same span receive `I-ENTITY`. Special tokens and padding receive `-100` so the loss ignores them.

Every row remains in its split. Text longer than `MAX_LENGTH` is truncated rather than silently removed.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if not tokenizer.is_fast:
    raise RuntimeError("A fast tokenizer is required for offset-based BIO alignment.")

print("Fast tokenizer loaded:", tokenizer.__class__.__name__)


In [ ]:
def tokenize_and_align_labels(example):
    encoded = tokenizer(
        example["source_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_attention_mask=True,
        return_offsets_mapping=True,
        return_token_type_ids=False,
    )

    bio_spans = privacy_mask_to_bio_spans(
        example["source_text"],
        example["privacy_mask"],
        example["uid"],
    )

    aligned_label_ids = []
    seen_entity_indices = set()

    for token_start, token_end in encoded["offset_mapping"]:
        if token_start == token_end:
            aligned_label_ids.append(-100)
            continue

        overlapping_entities = [
            (span_index, span)
            for span_index, span in enumerate(bio_spans)
            if token_start < span["end"] and token_end > span["start"]
        ]

        if len(overlapping_entities) > 1:
            raise ValueError(
                f"Row {example['uid']} has one tokenizer token overlapping "
                "multiple entity spans."
            )

        if not overlapping_entities:
            aligned_label_ids.append(label2id["O"])
            continue

        span_index, span = overlapping_entities[0]

        if span_index in seen_entity_indices:
            aligned_label_ids.append(label2id[span["inside_label"]])
        else:
            aligned_label_ids.append(label2id[span["begin_label"]])
            seen_entity_indices.add(span_index)

    encoded.pop("offset_mapping")
    encoded["labels"] = aligned_label_ids

    return encoded


Tokenize the three splits separately so their roles remain visible. These are full-dataset operations and may take time when executed.


In [ ]:
tokenized_train_dataset = project_train_dataset.map(
    tokenize_and_align_labels,
    remove_columns=project_train_dataset.column_names,
    desc="Tokenizing project training data",
)

tokenized_validation_dataset = project_validation_dataset.map(
    tokenize_and_align_labels,
    remove_columns=project_validation_dataset.column_names,
    desc="Tokenizing project validation data",
)

tokenized_final_test_dataset = final_test_dataset.map(
    tokenize_and_align_labels,
    remove_columns=final_test_dataset.column_names,
    desc="Tokenizing untouched final-test data",
)

print("Tokenization and BIO alignment completed for all three splits.")


#### 12. Inspect Token and BIO-Label Alignment

Inspect one annotated training example after tokenization. Only model tokens and their readable BIO labels are displayed; special tokens and padding are omitted.


In [ ]:
aligned_example = tokenize_and_align_labels(annotated_example)
aligned_tokens = tokenizer.convert_ids_to_tokens(aligned_example["input_ids"])

alignment_rows = [
    {
        "token": token,
        "BIO label": id2label[label_id],
    }
    for token, label_id in zip(aligned_tokens, aligned_example["labels"])
    if label_id != -100
]

pd.DataFrame(alignment_rows)


#### 13. Create DataLoaders

Create standard PyTorch DataLoaders. Training shuffles with a seeded generator, while validation and final testing preserve deterministic row order.


In [ ]:
from torch.utils.data import DataLoader

torch_columns = ["input_ids", "attention_mask", "labels"]

tokenized_train_dataset.set_format(type="torch", columns=torch_columns)
tokenized_validation_dataset.set_format(type="torch", columns=torch_columns)
tokenized_final_test_dataset.set_format(type="torch", columns=torch_columns)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_dataloader = DataLoader(
    tokenized_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
)

validation_dataloader = DataLoader(
    tokenized_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

final_test_dataloader = DataLoader(
    tokenized_final_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Training batches:   {len(train_dataloader):,}")
print(f"Validation batches: {len(validation_dataloader):,}")
print(f"Final-test batches: {len(final_test_dataloader):,}")


In [ ]:
shape_check_batch = next(iter(validation_dataloader))

print("input_ids:", tuple(shape_check_batch["input_ids"].shape))
print("attention_mask:", tuple(shape_check_batch["attention_mask"].shape))
print("labels:", tuple(shape_check_batch["labels"].shape))


#### 14. Load TinyBERT

Load a fresh pretrained TinyBERT with a newly initialized token-classification head. Rerunning this cell resets the model parameters; rerun Section 15 afterward to recreate loss, optimizer, and scheduler state.


In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    label2id=label2id,
    id2label=id2label,
)

model.to(device)

print(
    f"Fresh {MODEL_NAME} token-classification model loaded "
    f"with {len(label_list)} labels on {device}."
)


#### 15. Configure Loss and Optimizer

The default first experiment uses normal cross-entropy. Setting `USE_WEIGHTED_LOSS = True` gives `O` its configured weight and every entity-specific `B-` and `I-` label the configured PII weight.

Changing `USE_WEIGHTED_LOSS` defines a new experiment. Before retraining, reload a fresh model in Section 14 and rerun all three cells below so the loss, optimizer, and scheduler are recreated from the beginning.


In [ ]:
num_labels = len(label_list)

if USE_WEIGHTED_LOSS:
    class_weights = torch.full(
        (num_labels,),
        fill_value=PII_LABEL_WEIGHT,
        dtype=torch.float32,
        device=device,
    )
    class_weights[label2id["O"]] = O_LABEL_WEIGHT

    loss_function = torch.nn.CrossEntropyLoss(
        weight=class_weights,
        ignore_index=-100,
    )
    print("Weighted cross-entropy enabled.")
else:
    loss_function = torch.nn.CrossEntropyLoss(ignore_index=-100)
    print("Normal cross-entropy baseline enabled.")


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print(
    f"AdamW configured with lr={LEARNING_RATE} "
    f"and weight_decay={WEIGHT_DECAY}."
)


In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=LR_SCHEDULER_FACTOR,
    patience=LR_SCHEDULER_PATIENCE,
    min_lr=MIN_LEARNING_RATE,
)

print(
    "ReduceLROnPlateau configured "
    f"(enabled={USE_LR_SCHEDULER}, no warmup)."
)


#### 16. Train with Per-Epoch Validation

Train directly with PyTorch and validate after every completed epoch. Optimizer updates occur only in the training phase. Validation excludes `-100` positions, and the primary precision, recall, and F1 calculations exclude the `O` class.

`EPOCHS = 50` is the maximum. There is no automatic early stopping; inspect the summaries while training and stop manually if needed.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

history = {
    "train_loss": [],
    "validation_loss": [],
    "validation_precision": [],
    "validation_recall": [],
    "validation_f1": [],
    "validation_accuracy": [],
    "learning_rate": [],
}

for epoch in range(EPOCHS):
    # Training phase
    model.train()
    total_training_loss = 0.0

    for batch in train_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        logits = outputs.logits

        flattened_logits = logits.reshape(-1, num_labels)
        flattened_labels = labels.reshape(-1)

        loss = loss_function(flattened_logits, flattened_labels)
        loss.backward()
        optimizer.step()

        total_training_loss += loss.item()

    average_training_loss = total_training_loss / len(train_dataloader)

    # Validation phase
    model.eval()
    total_validation_loss = 0.0
    validation_true_labels = []
    validation_predicted_labels = []

    with torch.no_grad():
        for batch in validation_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            logits = outputs.logits

            flattened_logits = logits.reshape(-1, num_labels)
            flattened_labels = labels.reshape(-1)
            validation_loss = loss_function(flattened_logits, flattened_labels)

            total_validation_loss += validation_loss.item()

            predictions = logits.argmax(dim=-1)
            valid_positions = labels != -100

            validation_true_labels.extend(
                labels[valid_positions].detach().cpu().tolist()
            )
            validation_predicted_labels.extend(
                predictions[valid_positions].detach().cpu().tolist()
            )

    average_validation_loss = total_validation_loss / len(validation_dataloader)

    validation_precision, validation_recall, validation_f1, _ = (
        precision_recall_fscore_support(
            validation_true_labels,
            validation_predicted_labels,
            labels=pii_label_ids,
            average="micro",
            zero_division=0,
        )
    )
    validation_accuracy = accuracy_score(
        validation_true_labels,
        validation_predicted_labels,
    )

    if USE_LR_SCHEDULER:
        scheduler.step(average_validation_loss)

    current_learning_rate = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(average_training_loss)
    history["validation_loss"].append(average_validation_loss)
    history["validation_precision"].append(validation_precision)
    history["validation_recall"].append(validation_recall)
    history["validation_f1"].append(validation_f1)
    history["validation_accuracy"].append(validation_accuracy)
    history["learning_rate"].append(current_learning_rate)

    # Epoch summary
    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"train_loss: {average_training_loss:.4f} | "
        f"val_loss: {average_validation_loss:.4f} | "
        f"val_precision: {validation_precision:.4f} | "
        f"val_recall: {validation_recall:.4f} | "
        f"val_f1: {validation_f1:.4f} | "
        f"lr: {current_learning_rate:.2e}"
    )


#### 17. Final FP32 Test Evaluation

Evaluate the trained PyTorch FP32 model once on the untouched final-test split. The loop is explicit and performs no optimizer updates. PII precision, recall, and F1 ignore both `-100` positions and the `O` label; accuracy is retained only as a secondary measure.

Do not tune the experiment after inspecting these results without acknowledging that the split would no longer be an untouched final test.


In [ ]:
model.eval()
total_fp32_test_loss = 0.0
fp32_true_labels = []
fp32_predicted_labels = []

with torch.no_grad():
    for batch in final_test_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        logits = outputs.logits

        flattened_logits = logits.reshape(-1, num_labels)
        flattened_labels = labels.reshape(-1)
        test_loss = loss_function(flattened_logits, flattened_labels)

        total_fp32_test_loss += test_loss.item()

        predictions = logits.argmax(dim=-1)
        valid_positions = labels != -100

        fp32_true_labels.extend(labels[valid_positions].detach().cpu().tolist())
        fp32_predicted_labels.extend(
            predictions[valid_positions].detach().cpu().tolist()
        )

average_fp32_test_loss = total_fp32_test_loss / len(final_test_dataloader)

fp32_precision, fp32_recall, fp32_f1, _ = precision_recall_fscore_support(
    fp32_true_labels,
    fp32_predicted_labels,
    labels=pii_label_ids,
    average="micro",
    zero_division=0,
)
fp32_accuracy = accuracy_score(fp32_true_labels, fp32_predicted_labels)

(
    fp32_per_label_precision,
    fp32_per_label_recall,
    fp32_per_label_f1,
    fp32_per_label_support,
) = precision_recall_fscore_support(
    fp32_true_labels,
    fp32_predicted_labels,
    labels=pii_label_ids,
    average=None,
    zero_division=0,
)

fp32_per_label_metrics = {
    id2label[label_id]: {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "support": int(support),
    }
    for label_id, precision, recall, f1, support in zip(
        pii_label_ids,
        fp32_per_label_precision,
        fp32_per_label_recall,
        fp32_per_label_f1,
        fp32_per_label_support,
    )
}

fp32_metrics = {
    "framework": "PyTorch FP32",
    "average_loss": float(average_fp32_test_loss),
    "pii_precision": float(fp32_precision),
    "pii_recall": float(fp32_recall),
    "pii_f1": float(fp32_f1),
    "token_accuracy": float(fp32_accuracy),
    "per_label": fp32_per_label_metrics,
}

print(f"Final FP32 loss:      {average_fp32_test_loss:.4f}")
print(f"Final PII precision:  {fp32_precision:.4f}")
print(f"Final PII recall:     {fp32_recall:.4f}")
print(f"Final PII F1:         {fp32_f1:.4f}")
print(f"Token accuracy:       {fp32_accuracy:.4f} (secondary)")


In [ ]:
fp32_per_label_df = pd.DataFrame.from_dict(
    fp32_per_label_metrics,
    orient="index",
).reset_index(names="BIO label")

fp32_per_label_df


#### 18. Relevant Plots

Plot the completed-epoch history and final FP32 label results with simple Matplotlib figures.


In [ ]:
import matplotlib.pyplot as plt

completed_epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 4))
plt.plot(completed_epochs, history["train_loss"], marker="o", label="Training loss")
plt.plot(
    completed_epochs,
    history["validation_loss"],
    marker="o",
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Training and Validation Loss")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    completed_epochs,
    history["validation_precision"],
    marker="o",
    label="Precision",
)
plt.plot(
    completed_epochs,
    history["validation_recall"],
    marker="o",
    label="Recall",
)
plt.plot(
    completed_epochs,
    history["validation_f1"],
    marker="o",
    label="F1",
)
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.title("Validation PII Metrics")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
fp32_f1_plot_df = fp32_per_label_df.sort_values("f1")

plt.figure(figsize=(9, max(6, len(fp32_f1_plot_df) * 0.28)))
plt.barh(fp32_f1_plot_df["BIO label"], fp32_f1_plot_df["f1"])
plt.xlabel("F1")
plt.xlim(0, 1)
plt.title("Final FP32 Per-Label F1")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


#### 19. Example Predictions

Use a deterministic set of final-test rows so qualitative comparisons are reproducible. The reconstruction helper converts predicted BIO labels and tokenizer offsets into numbered entity placeholders.

Because inference is capped at `MAX_LENGTH`, any source suffix beyond the tokenizer's visible offsets remains unchanged in the displayed prediction.


In [ ]:
def reconstruct_masked_text(source_text, offset_mapping, predicted_label_ids):
    predicted_spans = []
    active_span = None

    for (token_start, token_end), predicted_label_id in zip(
        offset_mapping,
        predicted_label_ids,
    ):
        if token_start == token_end:
            continue

        predicted_label = id2label[int(predicted_label_id)]

        if predicted_label == "O":
            active_span = None
            continue

        bio_prefix, entity_type = predicted_label.split("-", maxsplit=1)

        starts_new_span = (
            bio_prefix == "B"
            or active_span is None
            or active_span["entity_type"] != entity_type
        )

        if starts_new_span:
            active_span = {
                "start": token_start,
                "end": token_end,
                "entity_type": entity_type,
            }
            predicted_spans.append(active_span)
        else:
            active_span["end"] = token_end

    masked_parts = []
    entity_counts = {}
    source_cursor = 0

    for span in predicted_spans:
        masked_parts.append(source_text[source_cursor:span["start"]])

        entity_type = span["entity_type"]
        entity_counts[entity_type] = entity_counts.get(entity_type, 0) + 1
        masked_parts.append(f"[{entity_type}_{entity_counts[entity_type]}]")

        source_cursor = span["end"]

    masked_parts.append(source_text[source_cursor:])

    return "".join(masked_parts), predicted_spans


In [ ]:
prediction_example_count = min(5, len(final_test_dataset))
prediction_rng = random.Random(SEED)
prediction_indices = prediction_rng.sample(
    range(len(final_test_dataset)),
    k=prediction_example_count,
)

prediction_rows = []
model.eval()

for example_index in prediction_indices:
    example = final_test_dataset[example_index]

    encoded_example = tokenizer(
        example["source_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_attention_mask=True,
        return_offsets_mapping=True,
        return_token_type_ids=False,
        return_tensors="pt",
    )

    offset_mapping = encoded_example.pop("offset_mapping")[0].tolist()
    model_inputs = {
        name: tensor.to(device)
        for name, tensor in encoded_example.items()
    }

    with torch.no_grad():
        prediction_logits = model(**model_inputs).logits

    predicted_label_ids = prediction_logits.argmax(dim=-1)[0].cpu().tolist()
    predicted_masked_text, _ = reconstruct_masked_text(
        example["source_text"],
        offset_mapping,
        predicted_label_ids,
    )

    prediction_rows.append({
        "Original source text": example["source_text"],
        "Expected masked text": example["masked_text"],
        "Predicted masked text": predicted_masked_text,
    })

pd.DataFrame(prediction_rows)


#### 20. Leakage-Safety Explanation

- Project validation was created only from the English original training split.
- The English original validation split was reserved as the untouched final test.
- Final-test rows were not used for training, optimizer updates, model selection, or hyperparameter tuning.
- `model.eval()` was used during validation and testing.
- `torch.no_grad()` was used during validation and testing.
- Optimizer updates occurred only inside the training batch loop.

The ontology check reads entity-type names from final test only to detect an unsupported label schema; it does not use final-test text, counts, losses, or metrics for training. After final-test results are inspected, changing the experiment based on them would compromise the split's final-test status.


#### 21. Save FP32 Artifacts and Export FP32 ONNX

Create the complete FP32 artifact tree automatically. Save the trained PyTorch model, tokenizer, mappings, training configuration, and PyTorch final-test metrics before exporting a single-file FP32 ONNX model.


In [ ]:
import json
from pathlib import Path

project_root = Path.cwd()
tinybert_directory = project_root / "model" / "TinyBERT-4"

if not tinybert_directory.is_dir():
    raise RuntimeError(
        "Run this notebook with the Home-AI repository root as the working directory."
    )

fp32_artifact_directory = tinybert_directory / "TinyBERT-4_FP32_artifacts"
fp32_huggingface_directory = fp32_artifact_directory / "huggingface_model"
fp32_tokenizer_directory = fp32_artifact_directory / "tokenizer"
fp32_onnx_directory = fp32_artifact_directory / "onnx_model"
fp32_onnx_path = fp32_onnx_directory / "model_fp32.onnx"

fp32_huggingface_directory.mkdir(parents=True, exist_ok=True)
fp32_tokenizer_directory.mkdir(parents=True, exist_ok=True)
fp32_onnx_directory.mkdir(parents=True, exist_ok=True)

model.save_pretrained(fp32_huggingface_directory)
tokenizer.save_pretrained(fp32_tokenizer_directory)

with (fp32_artifact_directory / "label2id.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(label2id, file, indent=2, ensure_ascii=False)

with (fp32_artifact_directory / "id2label.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        {str(label_id): label for label_id, label in id2label.items()},
        file,
        indent=2,
        ensure_ascii=False,
    )

training_config = {
    "dataset_name": DATASET_NAME,
    "model_name": MODEL_NAME,
    "seed": SEED,
    "validation_size": VALIDATION_SIZE,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "epochs_configured": EPOCHS,
    "epochs_completed": len(history["train_loss"]),
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "use_weighted_loss": USE_WEIGHTED_LOSS,
    "o_label_weight": O_LABEL_WEIGHT,
    "pii_label_weight": PII_LABEL_WEIGHT,
    "use_lr_scheduler": USE_LR_SCHEDULER,
    "lr_scheduler": "ReduceLROnPlateau",
    "lr_scheduler_factor": LR_SCHEDULER_FACTOR,
    "lr_scheduler_patience": LR_SCHEDULER_PATIENCE,
    "min_learning_rate": MIN_LEARNING_RATE,
    "final_learning_rate": (
        history["learning_rate"][-1]
        if history["learning_rate"]
        else optimizer.param_groups[0]["lr"]
    ),
    "split_protocol": {
        "project_train_and_validation": "English original train split",
        "final_test": "English original validation split",
    },
}

with (fp32_artifact_directory / "training_config.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(training_config, file, indent=2, ensure_ascii=False)

with (fp32_artifact_directory / "metrics_fp32.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump({"pytorch": fp32_metrics}, file, indent=2, ensure_ascii=False)

print("Saved FP32 PyTorch model, tokenizer, mappings, config, and metrics.")


Export only the model logits. The small wrapper makes the ONNX output explicit without changing training behavior. Dynamic batch and sequence dimensions support deployment inputs up to `MAX_LENGTH`.


In [ ]:
class TinyBertOnnxLogitsWrapper(torch.nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.trained_model = trained_model

    def forward(self, input_ids, attention_mask):
        return self.trained_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).logits


original_model_device = next(model.parameters()).device

try:
    model.to("cpu")
    model.eval()
    onnx_export_model = TinyBertOnnxLogitsWrapper(model).eval()

    onnx_example_inputs = tokenizer(
        [
            "TinyBERT ONNX export shape check.",
            "A second example keeps the exported batch dimension dynamic.",
        ],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_attention_mask=True,
        return_token_type_ids=False,
        return_tensors="pt",
    )

    batch_dimension = torch.export.Dim("batch_size", min=1)
    sequence_dimension = torch.export.Dim(
        "sequence_length",
        min=1,
        max=MAX_LENGTH,
    )

    onnx_program = torch.onnx.export(
        onnx_export_model,
        args=(
            onnx_example_inputs["input_ids"],
            onnx_example_inputs["attention_mask"],
        ),
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        opset_version=18,
        dynamo=True,
        external_data=False,
        dynamic_shapes=(
            {0: batch_dimension, 1: sequence_dimension},
            {0: batch_dimension, 1: sequence_dimension},
        ),
    )
    onnx_program.save(fp32_onnx_path, external_data=False)
finally:
    model.to(original_model_device)

print("FP32 ONNX model exported to:", fp32_onnx_path)


In [ ]:
import onnxruntime as ort

fp32_session_check = ort.InferenceSession(
    str(fp32_onnx_path),
    providers=["CPUExecutionProvider"],
)

print("ONNX Runtime session check: passed")
print("Inputs:", [model_input.name for model_input in fp32_session_check.get_inputs()])
print("Outputs:", [model_output.name for model_output in fp32_session_check.get_outputs()])


#### 22. Create INT8 Candidate

Create a post-training dynamic INT8 candidate from the completed FP32 ONNX model. This does not train the quantized model, overwrite FP32 files, or create the accepted INT8 artifact folder.


In [ ]:
from onnxruntime.quantization import QuantType, quantize_dynamic

int8_candidate_directory = tinybert_directory / "int8_candidate"
int8_candidate_directory.mkdir(parents=True, exist_ok=True)

int8_candidate_path = int8_candidate_directory / "model_int8.onnx"

quantize_dynamic(
    model_input=str(fp32_onnx_path),
    model_output=str(int8_candidate_path),
    weight_type=QuantType.QInt8,
    per_channel=False,
    reduce_range=False,
)

candidate_quantization_config = {
    "method": "ONNX Runtime dynamic post-training quantization",
    "weight_type": "QInt8",
    "per_channel": False,
    "reduce_range": False,
    "source_model": "model_fp32.onnx",
    "candidate_model": "model_int8.onnx",
    "onnxruntime_version": ort.__version__,
}

with (int8_candidate_directory / "quantization_config.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        candidate_quantization_config,
        file,
        indent=2,
        ensure_ascii=False,
    )

int8_session_check = ort.InferenceSession(
    str(int8_candidate_path),
    providers=["CPUExecutionProvider"],
)

print("INT8 candidate created and loaded successfully:", int8_candidate_path)


#### 23. Evaluate and Compare INT8

Create FP32 and INT8 ONNX Runtime sessions with identical CPU settings. Warm up both models on the same first final-test batches without timing them, then compare both models on every final-test example using the same preprocessing, `BATCH_SIZE`, and `MAX_LENGTH`.


In [ ]:
import time

fp32_session_options = ort.SessionOptions()
fp32_session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

int8_session_options = ort.SessionOptions()
int8_session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

fp32_onnx_session = ort.InferenceSession(
    str(fp32_onnx_path),
    sess_options=fp32_session_options,
    providers=["CPUExecutionProvider"],
)

int8_onnx_session = ort.InferenceSession(
    str(int8_candidate_path),
    sess_options=int8_session_options,
    providers=["CPUExecutionProvider"],
)

warmup_inputs = []

for batch_index, batch in enumerate(final_test_dataloader):
    if batch_index >= ONNX_WARMUP_BATCHES:
        break

    warmup_inputs.append({
        "input_ids": batch["input_ids"].cpu().numpy(),
        "attention_mask": batch["attention_mask"].cpu().numpy(),
    })

for model_inputs in warmup_inputs:
    fp32_onnx_session.run(None, model_inputs)

for model_inputs in warmup_inputs:
    int8_onnx_session.run(None, model_inputs)

print(
    f"Completed {len(warmup_inputs)} untimed warm-up batches "
    "for both FP32 and INT8."
)


In [ ]:
onnx_true_labels = []
fp32_onnx_predicted_labels = []
int8_onnx_predicted_labels = []
fp32_batch_latencies_seconds = []
int8_batch_latencies_seconds = []
onnx_example_count = 0

for batch in final_test_dataloader:
    model_inputs = {
        "input_ids": batch["input_ids"].cpu().numpy(),
        "attention_mask": batch["attention_mask"].cpu().numpy(),
    }
    labels_numpy = batch["labels"].cpu().numpy()
    onnx_example_count += model_inputs["input_ids"].shape[0]

    fp32_start_time = time.perf_counter()
    fp32_logits = fp32_onnx_session.run(None, model_inputs)[0]
    fp32_batch_latencies_seconds.append(time.perf_counter() - fp32_start_time)

    int8_start_time = time.perf_counter()
    int8_logits = int8_onnx_session.run(None, model_inputs)[0]
    int8_batch_latencies_seconds.append(time.perf_counter() - int8_start_time)

    fp32_predictions = fp32_logits.argmax(axis=-1)
    int8_predictions = int8_logits.argmax(axis=-1)
    valid_positions = labels_numpy != -100

    onnx_true_labels.extend(labels_numpy[valid_positions].tolist())
    fp32_onnx_predicted_labels.extend(
        fp32_predictions[valid_positions].tolist()
    )
    int8_onnx_predicted_labels.extend(
        int8_predictions[valid_positions].tolist()
    )

print(f"Timed final-test examples per model: {onnx_example_count:,}")


In [ ]:
onnx_metrics_by_model = {}

onnx_results = [
    (
        "FP32 ONNX",
        fp32_onnx_predicted_labels,
        fp32_batch_latencies_seconds,
        fp32_onnx_path,
    ),
    (
        "INT8 ONNX",
        int8_onnx_predicted_labels,
        int8_batch_latencies_seconds,
        int8_candidate_path,
    ),
]

for model_name, predictions, batch_latencies, model_path in onnx_results:
    precision, recall, f1, _ = precision_recall_fscore_support(
        onnx_true_labels,
        predictions,
        labels=pii_label_ids,
        average="micro",
        zero_division=0,
    )
    accuracy = accuracy_score(onnx_true_labels, predictions)

    per_label_precision, per_label_recall, per_label_f1, per_label_support = (
        precision_recall_fscore_support(
            onnx_true_labels,
            predictions,
            labels=pii_label_ids,
            average=None,
            zero_division=0,
        )
    )

    per_label_metrics = {
        id2label[label_id]: {
            "precision": float(label_precision),
            "recall": float(label_recall),
            "f1": float(label_f1),
            "support": int(label_support),
        }
        for label_id, label_precision, label_recall, label_f1, label_support in zip(
            pii_label_ids,
            per_label_precision,
            per_label_recall,
            per_label_f1,
            per_label_support,
        )
    }

    onnx_metrics_by_model[model_name] = {
        "pii_precision": float(precision),
        "pii_recall": float(recall),
        "pii_f1": float(f1),
        "token_accuracy": float(accuracy),
        "model_size_mb": float(model_path.stat().st_size / (1024 ** 2)),
        "average_latency_ms_per_example": float(
            1000 * sum(batch_latencies) / onnx_example_count
        ),
        "average_batch_latency_ms": float(1000 * np.mean(batch_latencies)),
        "median_batch_latency_ms": float(1000 * np.median(batch_latencies)),
        "timed_examples": int(onnx_example_count),
        "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH,
        "warmup_batches": ONNX_WARMUP_BATCHES,
        "execution_provider": "CPUExecutionProvider",
        "per_label": per_label_metrics,
    }

fp32_onnx_metrics = onnx_metrics_by_model["FP32 ONNX"]
int8_metrics = onnx_metrics_by_model["INT8 ONNX"]


Present the FP32 and INT8 quality, size, and latency results together. All signed differences below are calculated as INT8 minus FP32.


In [ ]:
onnx_comparison = {
    "difference_definition": "INT8 minus FP32",
    "precision_difference": (
        int8_metrics["pii_precision"] - fp32_onnx_metrics["pii_precision"]
    ),
    "recall_difference": (
        int8_metrics["pii_recall"] - fp32_onnx_metrics["pii_recall"]
    ),
    "f1_difference": int8_metrics["pii_f1"] - fp32_onnx_metrics["pii_f1"],
    "model_size_difference_mb": (
        int8_metrics["model_size_mb"] - fp32_onnx_metrics["model_size_mb"]
    ),
    "average_latency_difference_ms_per_example": (
        int8_metrics["average_latency_ms_per_example"]
        - fp32_onnx_metrics["average_latency_ms_per_example"]
    ),
}

comparison_df = pd.DataFrame(
    [
        {
            "Model": model_name,
            "PII precision": metrics["pii_precision"],
            "PII recall": metrics["pii_recall"],
            "PII F1": metrics["pii_f1"],
            "Size (MB)": metrics["model_size_mb"],
            "Average latency (ms/example)": metrics[
                "average_latency_ms_per_example"
            ],
            "Median batch latency (ms)": metrics["median_batch_latency_ms"],
        }
        for model_name, metrics in onnx_metrics_by_model.items()
    ]
)

comparison_df


In [ ]:
int8_per_label_df = pd.DataFrame.from_dict(
    int8_metrics["per_label"],
    orient="index",
).reset_index(names="BIO label")

print("INT8 minus FP32 differences:")
for metric_name, difference in onnx_comparison.items():
    if metric_name != "difference_definition":
        print(f"  {metric_name}: {difference:.6f}")

int8_per_label_df


In [ ]:
with (fp32_artifact_directory / "metrics_fp32.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        {
            "pytorch": fp32_metrics,
            "onnxruntime": fp32_onnx_metrics,
        },
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Updated metrics_fp32.json with FP32 ONNX Runtime results.")


#### 24. Save Accepted INT8 Artifacts

Run the next cell only if the INT8 quality, size, and latency trade-off is acceptable. It creates an independent deployment folder, copies no FP32 PyTorch weights, and leaves `int8_candidate` intact.

The selected deployment approach is direct ONNX Runtime inference. Therefore, `huggingface_model` contains the quantized ONNX weights plus the Hugging Face model configuration, while the sibling `tokenizer` folder contains the complete tokenizer.


In [ ]:
import shutil

int8_artifact_directory = tinybert_directory / "TinyBERT-4_INT8_artifacts"
int8_huggingface_directory = int8_artifact_directory / "huggingface_model"
int8_tokenizer_directory = int8_artifact_directory / "tokenizer"
int8_final_model_path = int8_huggingface_directory / "model_int8.onnx"

int8_huggingface_directory.mkdir(parents=True, exist_ok=True)
int8_tokenizer_directory.mkdir(parents=True, exist_ok=True)

shutil.copy2(int8_candidate_path, int8_final_model_path)
model.config.save_pretrained(int8_huggingface_directory)
tokenizer.save_pretrained(int8_tokenizer_directory)

with (int8_artifact_directory / "label2id.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(label2id, file, indent=2, ensure_ascii=False)

with (int8_artifact_directory / "id2label.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        {str(label_id): label for label_id, label in id2label.items()},
        file,
        indent=2,
        ensure_ascii=False,
    )

accepted_quantization_config = {
    **candidate_quantization_config,
    "accepted_model": "huggingface_model/model_int8.onnx",
    "inference_api": "onnxruntime.InferenceSession",
    "tokenizer_directory": "tokenizer",
}

with (int8_artifact_directory / "quantization_config.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        accepted_quantization_config,
        file,
        indent=2,
        ensure_ascii=False,
    )

with (int8_artifact_directory / "metrics_int8.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        {
            "int8_onnxruntime": int8_metrics,
            "comparison_with_fp32_onnx": onnx_comparison,
        },
        file,
        indent=2,
        ensure_ascii=False,
    )

for forbidden_weight_file in ["model.safetensors", "pytorch_model.bin"]:
    forbidden_path = int8_huggingface_directory / forbidden_weight_file
    if forbidden_path.exists():
        raise RuntimeError(
            f"Unexpected FP32 weight file in INT8 artifacts: {forbidden_path}"
        )

accepted_int8_session = ort.InferenceSession(
    str(int8_final_model_path),
    providers=["CPUExecutionProvider"],
)
accepted_tokenizer = AutoTokenizer.from_pretrained(
    int8_tokenizer_directory,
    local_files_only=True,
    use_fast=True,
)

print("Accepted INT8 artifacts saved and reopened successfully:")
print(int8_artifact_directory)
print("Model inputs:", [item.name for item in accepted_int8_session.get_inputs()])
print("Tokenizer:", accepted_tokenizer.__class__.__name__)
